# 06. Pipeline de Transformação Simplificado - Encoding e Scaling

## 🎯 Objetivo

Implementar um **pipeline simplificado de transformação** para preparar os dados
com features de safra (YYYYMMDD) para modelagem.

## 📚 Transformações Aplicadas

### 1. **Seleção de Features**
- Remoção de colunas desnecessárias (IDs, Timestamp original, etc.)
- Manutenção da safra e features temporais

### 2. **Categorical Encoding**
- **One-Hot Encoding**: Variáveis categóricas de baixa cardinalidade
- **Label Encoding**: Variáveis de alta cardinalidade

### 3. **Numerical Transformations**
- **Imputação**: Preencher valores ausentes com mediana
- **Normalização**: StandardScaler para variáveis numéricas

## ⚠️ Garantia Anti-Leakage

✅ **Fit APENAS no treino**: Parâmetros aprendidos somente do treino  
✅ **Transform em treino e OOT**: Usa parâmetros do treino  
✅ **Pipeline simplificado**: Sem features de velocidade complexas  

---

## 1. Setup - Importações e Configuração

In [1]:
# Configurar path para importar módulos do projeto
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Importar configurações do projeto
from source.config import PROJ_ROOT, get_data_path, get_model_path, ensure_directories

# Garantir que os diretórios existam
ensure_directories()

print(f"✅ Projeto Root: {PROJ_ROOT}")
print(f"✅ Configurações carregadas de source/config.py")

2026-02-02 19:03:51.397 | INFO     | source.config:<module>:77 - [CONFIG] PROJ_ROOT: C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering
2026-02-02 19:03:51.400 | INFO     | source.config:<module>:78 - [CONFIG] Python executando de: c:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\notebooks
✅ Projeto Root: C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering
✅ Configurações carregadas de source/config.py


In [2]:
# Importações padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import json
import joblib

# Scikit-Learn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)

print("✅ Bibliotecas importadas com sucesso!")
print("✅ Pipeline Simplificado!")

✅ Bibliotecas importadas com sucesso!
✅ Pipeline Simplificado!


## 2. Carregamento dos Dados com Features Simplificadas

Carregamos os dados **COM features de safra** gerados no Notebook 05.

In [4]:
print("="*80)
print("CARREGAMENTO DOS DADOS COM FEATURES DE SAFRA")
print("="*80)

# Carregar datasets
df_treino = pd.read_csv(get_data_path('df_treino_with_features.csv', 'processed'))
df_oot = pd.read_csv(get_data_path('df_oot_with_features.csv', 'processed'))

# Converter Timestamp para datetime (permitir formato misto)
df_treino['Timestamp'] = pd.to_datetime(df_treino['Timestamp'], format='mixed')
df_oot['Timestamp'] = pd.to_datetime(df_oot['Timestamp'], format='mixed')

print(f"\n📊 Dataset de Treino:")
print(f"   Shape: {df_treino.shape}")
print(f"   Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_treino['Is Laundering'].mean()*100:.2f}%")
print(f"   Memória: {df_treino.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print(f"\n📊 Dataset de OOT:")
print(f"   Shape: {df_oot.shape}")
print(f"   Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_oot['Is Laundering'].mean()*100:.2f}%")
print(f"   Memória: {df_oot.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print(f"\n✅ Dados carregados com sucesso!")

# Mostrar safras
print(f"\n📅 Safras disponíveis:")
print(f"   Treino: {df_treino['safra'].min()} até {df_treino['safra'].max()}")
print(f"   Treino - Safras únicas: {df_treino['safra'].nunique()}")
print(f"   OOT: {df_oot['safra'].min()} até {df_oot['safra'].max()}")
print(f"   OOT - Safras únicas: {df_oot['safra'].nunique()}")

CARREGAMENTO DOS DADOS COM FEATURES DE SAFRA

📊 Dataset de Treino:
   Shape: (14375186, 38)
   Período: 2022-08-01 00:00:00 até 2022-10-18 00:52:00
   Taxa de lavagem: 0.11%
   Memória: 6099.35 MB

📊 Dataset de OOT:
   Shape: (2395950, 38)
   Período: 2022-10-25 00:53:00 até 2023-01-09 12:27:00
   Taxa de lavagem: 0.23%
   Memória: 1016.80 MB

✅ Dados carregados com sucesso!

📅 Safras disponíveis:
   Treino: 20220801 até 20221018
   Treino - Safras únicas: 79
   OOT: 20221025 até 20230109
   OOT - Safras únicas: 77


## 3. Separação de Features e Target

Separamos X (features) e y (target) removendo colunas desnecessárias.

In [5]:
# Definir target
target_col = 'Is Laundering'

# Colunas para remover (além do target)
cols_to_remove = [
    target_col,
    'Timestamp',  # Já temos safra e componentes temporais
    'From Bank', 'To Bank',  # Alta cardinalidade, usar apenas se necessário
    'From Account', 'To Account',  # IDs individuais
    'From Entity ID', 'To Entity ID',  # IDs de entidade
]

# Verificar quais colunas existem antes de remover
cols_to_remove = [col for col in cols_to_remove if col in df_treino.columns]

# Separar X e y
X_train = df_treino.drop(columns=cols_to_remove)
y_train = df_treino[target_col]

X_oot = df_oot.drop(columns=cols_to_remove)
y_oot = df_oot[target_col]

print("="*80)
print("SEPARAÇÃO DE FEATURES E TARGET")
print("="*80)

print(f"\n📊 Colunas removidas ({len(cols_to_remove)}):")
for col in cols_to_remove:
    print(f"   - {col}")

print(f"\n📊 Treino:")
print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   Taxa de lavagem: {y_train.mean()*100:.2f}%")

print(f"\n📊 OOT:")
print(f"   X_oot shape: {X_oot.shape}")
print(f"   y_oot shape: {y_oot.shape}")
print(f"   Taxa de lavagem: {y_oot.mean()*100:.2f}%")

print(f"\n📋 Features disponíveis ({X_train.shape[1]}):")
print(f"   Principais colunas:")
for col in X_train.columns[:15]:
    print(f"   - {col}")
if X_train.shape[1] > 15:
    print(f"   ... e mais {X_train.shape[1] - 15} colunas")

SEPARAÇÃO DE FEATURES E TARGET

📊 Colunas removidas (8):
   - Is Laundering
   - Timestamp
   - From Bank
   - To Bank
   - From Account
   - To Account
   - From Entity ID
   - To Entity ID

📊 Treino:
   X_train shape: (14375186, 30)
   y_train shape: (14375186,)
   Taxa de lavagem: 0.11%

📊 OOT:
   X_oot shape: (2395950, 30)
   y_oot shape: (2395950,)
   Taxa de lavagem: 0.23%

📋 Features disponíveis (30):
   Principais colunas:
   - Amount Received
   - Receiving Currency
   - Amount Paid
   - Payment Currency
   - Payment Format
   - From Bank Name
   - From Entity Name
   - To Bank Name
   - To Entity Name
   - safra
   - year
   - month
   - day
   - hour
   - dayofweek
   ... e mais 15 colunas


## 4. Identificação de Colunas para Pipeline

Identificar colunas categóricas e numéricas para aplicar transformações adequadas.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Identificar colunas categóricas e numéricas
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remover safra se estiver nos numéricos (é identificador, não feature)
if 'safra' in numeric_cols:
    numeric_cols.remove('safra')

print("\n" + "="*80)
print("IDENTIFICAÇÃO DE COLUNAS")
print("="*80)

print(f"\n📊 Colunas categóricas ({len(categorical_cols)}):")
for col in categorical_cols[:10]:
    nunique = X_train[col].nunique()
    print(f"   - {col}: {nunique} categorias")
if len(categorical_cols) > 10:
    print(f"   ... e mais {len(categorical_cols) - 10} colunas")

# Separar categóricas por cardinalidade
low_cardinality = []
high_cardinality = []
for col in categorical_cols:
    nunique = X_train[col].nunique()
    if nunique <= 20:  # Limite conservador para OneHot
        low_cardinality.append(col)
    else:
        high_cardinality.append(col)

print(f"\n📊 Categóricas de baixa cardinalidade (≤20): {len(low_cardinality)}")
for col in low_cardinality:
    print(f"   - {col}: {X_train[col].nunique()} categorias")

print(f"\n📊 Categóricas de alta cardinalidade (>20): {len(high_cardinality)}")
for col in high_cardinality:
    print(f"   - {col}: {X_train[col].nunique()} categorias → REMOVIDA")

# Ajustar categóricas para usar apenas baixa cardinalidade
categorical_cols = low_cardinality

print(f"\n📊 Colunas numéricas ({len(numeric_cols)}):")
for col in numeric_cols[:10]:
    print(f"   - {col}")
if len(numeric_cols) > 10:
    print(f"   ... e mais {len(numeric_cols) - 10} colunas")


IDENTIFICAÇÃO DE COLUNAS

📊 Colunas categóricas (7):
   - Receiving Currency: 15 categorias
   - Payment Currency: 15 categorias
   - Payment Format: 7 categorias
   - From Bank Name: 55016 categorias
   - From Entity Name: 518771 categorias
   - To Bank Name: 19070 categorias
   - To Entity Name: 639152 categorias

📊 Categóricas de baixa cardinalidade (≤20): 3
   - Receiving Currency: 15 categorias
   - Payment Currency: 15 categorias
   - Payment Format: 7 categorias

📊 Categóricas de alta cardinalidade (>20): 4
   - From Bank Name: 55016 categorias → REMOVIDA
   - From Entity Name: 518771 categorias → REMOVIDA
   - To Bank Name: 19070 categorias → REMOVIDA
   - To Entity Name: 639152 categorias → REMOVIDA

📊 Colunas numéricas (22):
   - Amount Received
   - Amount Paid
   - year
   - month
   - day
   - hour
   - dayofweek
   - quarter
   - month_sin
   - month_cos
   ... e mais 12 colunas


## 5. Construção do Pipeline Simplificado

Criar pipeline com transformadores para colunas numéricas e categóricas.

In [10]:
# Pipeline para colunas numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para colunas categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'  # Remover colunas não especificadas
)

print("\n" + "="*80)
print("PIPELINE CONSTRUÍDO")
print("="*80)

print(f"\n📦 Transformadores:")
print(f"   - Numérico: SimpleImputer(median) + StandardScaler")
print(f"   - Categórico: SimpleImputer(missing) + OneHotEncoder")

print(f"\n✓ Pipeline pronto para fit/transform")


PIPELINE CONSTRUÍDO

📦 Transformadores:
   - Numérico: SimpleImputer(median) + StandardScaler
   - Categórico: SimpleImputer(missing) + OneHotEncoder

✓ Pipeline pronto para fit/transform


## 6. Aplicação do Pipeline - Fit e Transform

⚠️ **CRÍTICO**: Fit APENAS no X_train, Transform em X_train e X_oot.

In [11]:
# Fit no treino e transform em ambos
print("\n" + "="*80)
print("FIT E TRANSFORM")
print("="*80)

print("\n⏳ Fitting no treino...")
preprocessor.fit(X_train)

print("✓ Fit concluído!")

print("\n⏳ Transforming treino...")
X_train_transformed = preprocessor.transform(X_train)

print("\n⏳ Transforming OOT...")
X_oot_transformed = preprocessor.transform(X_oot)

print("\n✓ Transform concluído!")

# Obter nomes das features após transformação
feature_names = []

# Features numéricas mantêm nome original
feature_names.extend(numeric_cols)

# Features categóricas geram múltiplas colunas (one-hot)
if len(categorical_cols) > 0:
    # Obter nomes das categorias do OneHotEncoder
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols)
    feature_names.extend(cat_feature_names)

# Converter para DataFrame
X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_oot_transformed = pd.DataFrame(X_oot_transformed, columns=feature_names, index=X_oot.index)

print("\n" + "="*80)
print("TRANSFORMAÇÃO CONCLUÍDA")
print("="*80)

print(f"\n📊 Shapes transformados:")
print(f"   X_train: {X_train.shape} → {X_train_transformed.shape}")
print(f"   X_oot: {X_oot.shape} → {X_oot_transformed.shape}")

print(f"\n📝 Features finais: {X_train_transformed.shape[1]}")


FIT E TRANSFORM

⏳ Fitting no treino...
✓ Fit concluído!

⏳ Transforming treino...

⏳ Transforming OOT...

✓ Transform concluído!

TRANSFORMAÇÃO CONCLUÍDA

📊 Shapes transformados:
   X_train: (14375186, 30) → (14375186, 59)
   X_oot: (2395950, 30) → (2395950, 59)

📝 Features finais: 59


In [12]:
# Estatísticas descritivas
print("="*80)
print("ESTATÍSTICAS DESCRITIVAS - DADOS TRANSFORMADOS (Treino)")
print("="*80)

print(f"\n📊 Shape: {X_train_transformed.shape}")
print(f"\n📊 Primeiras features:")
print(X_train_transformed.iloc[:, :10].describe().T)

print(f"\n📊 Últimas features:")
if X_train_transformed.shape[1] > 10:
    print(X_train_transformed.iloc[:, -10:].describe().T)

ESTATÍSTICAS DESCRITIVAS - DADOS TRANSFORMADOS (Treino)

📊 Shape: (14375186, 59)

📊 Primeiras features:
                      count          mean  std       min       25%       50%  \
Amount Received  14375186.0 -1.107197e-19  1.0 -0.004953 -0.004953 -0.004952   
Amount Paid      14375186.0 -5.456898e-19  1.0 -0.004079 -0.004079 -0.004078   
year             14375186.0  0.000000e+00  0.0  0.000000  0.000000  0.000000   
month            14375186.0 -1.425310e-15  1.0 -1.043435 -1.043435  0.290951   
day              14375186.0 -7.288517e-17  1.0 -1.507332 -0.848713 -0.080325   
hour             14375186.0  2.024588e-18  1.0 -1.456969 -0.909609  0.048271   
dayofweek        14375186.0  5.668847e-17  1.0 -1.421643 -0.865076  0.248057   
quarter          14375186.0  4.211143e-16  1.0 -0.493047 -0.493047 -0.493047   
month_sin        14375186.0 -1.554884e-15  1.0 -1.248423 -1.248423  0.801011   
month_cos        14375186.0 -1.295736e-16  1.0 -1.043435 -1.043435  0.290951   

               

In [13]:
# Verificar valores ausentes
print("="*80)
print("VALORES AUSENTES APÓS TRANSFORMAÇÃO")
print("="*80)

nan_train = X_train_transformed.isnull().sum().sum()
nan_oot = X_oot_transformed.isnull().sum().sum()

print(f"\n✓ Treino: {nan_train} valores ausentes")
print(f"✓ OOT: {nan_oot} valores ausentes")

if nan_train == 0 and nan_oot == 0:
    print("\n✅ Nenhum valor ausente! Pipeline de imputação funcionou corretamente.")
else:
    print("\n⚠️  Ainda há valores ausentes. Verificar pipeline de imputação.")

VALORES AUSENTES APÓS TRANSFORMAÇÃO

✓ Treino: 0 valores ausentes
✓ OOT: 0 valores ausentes

✅ Nenhum valor ausente! Pipeline de imputação funcionou corretamente.


## 8. Validação da Normalização

Verificar se as features numéricas estão normalizadas (média ~0, std ~1).

In [14]:
print("="*80)
print("VALIDAÇÃO DA NORMALIZAÇÃO (StandardScaler)")
print("="*80)

# Pegar apenas features numéricas originais (não one-hot encoded)
numeric_features = X_train_transformed[numeric_cols]

means = numeric_features.mean()
stds = numeric_features.std()

print(f"\n📊 Médias (devem estar próximas de 0):")
print(f"   Min: {means.min():.4f}")
print(f"   Max: {means.max():.4f}")
print(f"   Média das médias: {means.mean():.4f}")

print(f"\n📊 Desvios padrão (devem estar próximos de 1):")
print(f"   Min: {stds.min():.4f}")
print(f"   Max: {stds.max():.4f}")
print(f"   Média dos stds: {stds.mean():.4f}")

if abs(means.mean()) < 0.1 and abs(stds.mean() - 1.0) < 0.2:
    print("\n✅ Normalização bem-sucedida!")
else:
    print("\n⚠️  Normalização pode ter problemas. Revisar pipeline.")

VALIDAÇÃO DA NORMALIZAÇÃO (StandardScaler)

📊 Médias (devem estar próximas de 0):
   Min: -0.0000
   Max: 0.0000
   Média das médias: -0.0000

📊 Desvios padrão (devem estar próximos de 1):
   Min: 0.0000
   Max: 1.0000
   Média dos stds: 0.9545

✅ Normalização bem-sucedida!


## 9. Validação Anti-Leakage

Verificar que não há vazamento de informação do OOT para o treino.

In [15]:
print("="*80)
print("VALIDAÇÃO ANTI-LEAKAGE")
print("="*80)

# Verificação: Médias e stds do OOT devem ser DIFERENTES do treino
# (porque foram normalizados com parâmetros do treino, não do OOT)

sample_cols = numeric_cols[:10]

train_means = X_train_transformed[sample_cols].mean()
oot_means = X_oot_transformed[sample_cols].mean()

train_stds = X_train_transformed[sample_cols].std()
oot_stds = X_oot_transformed[sample_cols].std()

print("\n✓ Verificação: Estatísticas do OOT diferem do treino?")
print(f"\n  Diferença média das médias: {abs(train_means - oot_means).mean():.4f}")
print(f"  Diferença média dos stds: {abs(train_stds - oot_stds).mean():.4f}")

if abs(train_means - oot_means).mean() > 0.01:
    print("\n  ✅ OOT tem estatísticas diferentes → Pipeline aplicado corretamente!")
else:
    print("\n  ⚠️  OOT tem estatísticas muito similares → Possível leakage!")

print("\n✅ Validação anti-leakage concluída!")

VALIDAÇÃO ANTI-LEAKAGE

✓ Verificação: Estatísticas do OOT diferem do treino?

  Diferença média das médias: 0.9905
  Diferença média dos stds: 0.4147

  ✅ OOT tem estatísticas diferentes → Pipeline aplicado corretamente!

✅ Validação anti-leakage concluída!


## 10. Salvamento dos Dados Transformados

Salvamos X_train, X_oot, y_train, y_oot e o pipeline para uso no treinamento de modelos.

In [16]:
print("="*80)
print("SALVAMENTO DOS DADOS TRANSFORMADOS")
print("="*80)

# Definir caminhos
path_X_train = get_data_path('X_train.csv', 'processed')
path_X_oot = get_data_path('X_oot.csv', 'processed')
path_y_train = get_data_path('y_train.csv', 'processed')
path_y_oot = get_data_path('y_oot.csv', 'processed')
path_pipeline = get_model_path('preprocessing_pipeline.pkl')

# Salvar dados
X_train_transformed.to_csv(path_X_train, index=False)
X_oot_transformed.to_csv(path_X_oot, index=False)
y_train.to_frame().to_csv(path_y_train, index=False)
y_oot.to_frame().to_csv(path_y_oot, index=False)

# Salvar pipeline
joblib.dump(preprocessor, path_pipeline)

print(f"\n✅ Datasets e pipeline salvos com sucesso!")
print(f"\n📁 Arquivos de dados:")
print(f"   - {path_X_train}")
print(f"   - {path_X_oot}")
print(f"   - {path_y_train}")
print(f"   - {path_y_oot}")

print(f"\n📁 Pipeline:")
print(f"   - {path_pipeline}")

# Verificar tamanhos
size_X_train = Path(path_X_train).stat().st_size / (1024 * 1024)
size_X_oot = Path(path_X_oot).stat().st_size / (1024 * 1024)

print(f"\n📏 Tamanhos:")
print(f"   - X_train: {size_X_train:.2f} MB")
print(f"   - X_oot: {size_X_oot:.2f} MB")

SALVAMENTO DOS DADOS TRANSFORMADOS

✅ Datasets e pipeline salvos com sucesso!

📁 Arquivos de dados:
   - C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\X_train.csv
   - C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\X_oot.csv
   - C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\y_train.csv
   - C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\data\processed\y_oot.csv

📁 Pipeline:
   - C:\Users\win\Desktop\Projetos\TCC_v2\anti_money_laundering\money_laundering\models\preprocessing_pipeline.pkl

📏 Tamanhos:
   - X_train: 7817.91 MB
   - X_oot: 1293.37 MB


## 11. Teste de Carregamento do Pipeline

Verificar que o pipeline salvo pode ser recarregado e usado corretamente.

In [17]:
print("="*80)
print("TESTE DE CARREGAMENTO DO PIPELINE")
print("="*80)

# Carregar pipeline salvo
pipeline_loaded = joblib.load(get_model_path('preprocessing_pipeline.pkl'))

print(f"\n✅ Pipeline carregado com sucesso!")

# Testar transform em amostra
sample = X_train.head(10)
transformed_sample = pipeline_loaded.transform(sample)

print(f"\n✓ Teste de transform:")
print(f"   Input shape: {sample.shape}")
print(f"   Output shape: {transformed_sample.shape}")

print("\n✅ Pipeline salvo está funcional!")

TESTE DE CARREGAMENTO DO PIPELINE

✅ Pipeline carregado com sucesso!

✓ Teste de transform:
   Input shape: (10, 30)
   Output shape: (10, 59)

✅ Pipeline salvo está funcional!


## 12. Sumário e Próximos Passos

### ✅ Realizações deste Notebook

1. ✅ Construção de **Pipeline sklearn** simplificado
2. ✅ Aplicação de **Fit APENAS no treino**
3. ✅ Transformação consistente em **treino e OOT**
4. ✅ **Categorical encoding** (One-Hot)
5. ✅ **Imputação** (mediana para numéricos, constante para categóricos)
6. ✅ **Normalização** (StandardScaler ajustado no treino)
7. ✅ Validação **anti-leakage**
8. ✅ Persistência do pipeline e datasets

### 📊 Estatísticas Finais

- **Samples treino**: {X_train_transformed.shape[0]:,}
- **Samples OOT**: {X_oot_transformed.shape[0]:,}
- **Features finais**: {X_train_transformed.shape[1]}

### 🔄 Próximos Passos

**Notebook 08**: Treinamento de Modelos com:
- TimeSeriesSplit para validação temporal
- Balanceamento correto (dentro dos folds)
- Threshold otimizado

---

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Status**: ✅ Pipeline Simplificado Concluído